<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">میان‌بر دوم به کجا وصل می‌شود؟</h1>
<p style="text-align:right">درس 46 از 92 · یک بلوک را از قطعه‌های آشنا بسازیم · <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">40-block</code></p>
<p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-04/40-block.html">📖 بازگشت به همین درس</a></p>
<p style="text-align:right"><bdi dir="ltr">Pre-Norm</bdi> واقعی را بازسازی کنید و خطایی را بگیرید که <bdi dir="ltr">Shape</bdi> را عوض نمی‌کند.</p><p style="text-align:right">پیش‌نیاز: <bdi dir="ltr">Multi-Head Attention</bdi>، <bdi dir="ltr">FFN</bdi>، <bdi dir="ltr">Residual</bdi> و <bdi dir="ltr">Layer Normalization</bdi> را بشناسید.</p>
<p style="text-align:right">زمان یادگیری درس همراه با همین دفتر: حدود ۹۰–۱۴۵ دقیقه. زمان دفتر دوباره به زمان درس اضافه نمی‌شود؛ نصب و تمرین اختیاری جداست.</p>
<p style="text-align:right">این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو <bdi dir="ltr">Cell</bdi> با برچسب <bdi dir="ltr">TODO</bdi> را خودتان کامل کنید. پیام <bdi dir="ltr">INCOMPLETE</bdi> یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p style="text-align:right">از بالا به پایین اجرا کنید. پس از تغییر هر تابع، <bdi dir="ltr">Cell</bdi> آن و سپس <bdi dir="ltr">Cell</bdi> آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code style="direction:ltr;text-align:left;unicode-bidi:isolate">Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">اگر <bdi dir="ltr">Attention</bdi> اصلاحی به <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">x</code> اضافه کند، <bdi dir="ltr">FFN</bdi> باید کدام نمایش را بگیرد و جمع دوم باید با کدام مقدار انجام شود؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
from mini_gpt.config import ModelConfig
from mini_gpt.transformer import TransformerBlock
block = TransformerBlock(ModelConfig(12,8,8,2,1,0.)).eval()
x = torch.randn(2,5,8)
trace = {}
with torch.no_grad():
    expected = block(x,trace=trace)
print('block input/output:',x.shape,expected.shape)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right">تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">manual_block(block,x)</code> زوج <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(after_attention, output)</code> برگرداند. زیرلایه‌های خود <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">block</code> را به کار ببرید، ولی <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">block(x)</code> را صدا نزنید. ترتیب <bdi dir="ltr">Pre-Norm</bdi> و هر دو <bdi dir="ltr">Residual</bdi> را حفظ کنید.</p>
</div>

In [ ]:
def manual_block(block, x):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = manual_block(block,x)
    if result is None: return False
    after_attention,out = result
    torch.testing.assert_close(out,expected)
    torch.testing.assert_close(after_attention,x+block.attention(block.norm_1(x)))
    torch.testing.assert_close(out-after_attention,trace['feed_forward'])
    y = torch.randn(1,3,8)
    torch.testing.assert_close(manual_block(block,y)[1],block(y))
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">residual</code> را روی همان بلوک خاموش کنید. در این <bdi dir="ltr">API</bdi> هر دو جمع حذف می‌شوند؛ <bdi dir="ltr">Layer Normalization</bdi> و <bdi dir="ltr">Mask</bdi> باقی می‌مانند. از این اختلاف عددی به‌تنهایی نمی‌توان دربارهٔ کیفیت آموزش نتیجه گرفت.</p>
</div>

In [ ]:
with torch.no_grad():
    without = block(x,residual=False)
    manual_without = block.feed_forward(block.norm_2(block.attention(block.norm_1(x))))
    torch.testing.assert_close(without,manual_without)
    print('same shape:',without.shape,'difference:',(without-expected).abs().mean().item())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">نسخهٔ خراب جمع دوم را دوباره به ورودی آغاز بلوک وصل می‌کند. تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">repair_block(block,x)</code> فقط خروجی نهایی صحیح را برگرداند.</p>
</div>

In [ ]:
with torch.no_grad():
    middle = x+block.attention(block.norm_1(x))
    wrong = x+block.feed_forward(block.norm_2(middle))
print('same shape:',wrong.shape,'wrong skip difference:',(wrong-expected).abs().max().item())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def repair_block(block, x):
    # TODO
    return None

In [ ]:
def test_repair():
    result = repair_block(block,x)
    if result is None: return False
    torch.testing.assert_close(result,expected)
    for T in (1,4):
        y = torch.randn(1,T,8)
        torch.testing.assert_close(repair_block(block,y),block(y))
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">TransformerBlock</code> واقعی هم در <bdi dir="ltr">v5</bdi> و هم در <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">MiniGPT</code> استفاده می‌شود. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">trace[&#x27;feed_forward&#x27;]</code> اصلاح <bdi dir="ltr">FFN</bdi> پیش از جمع است، نه خروجی نهایی بلوک؛ این تفاوت در آزمون میانی سنجیده شد.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">کدام مقایسه نشان داد بلوک فقط هم‌شکل نیست، بلکه همان محاسبهٔ مرجع را انجام می‌دهد؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-04/40-block.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/40-block.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>